# Baseline YOLO model

### Train a baseline YOLO model on the combined mapillary and LISA signs

In [3]:
import os
import csv
import json
import yaml
import shutil
import random
from pathlib import Path
from collections import defaultdict
from PIL import Image
from tqdm.notebook import tqdm

## Configuration

In [4]:
# ── Inputs ────────────────────────────────────────────────────────────────────
MAPILLARY_ROOT   = Path("dataset/mtsd_v2_fully_annotated")
LISA_ROOT        = Path("dataset/lisa")
MAPILLARY_CSV    = Path("mapillary_class_review.csv")   # from mapillary review notebook
LISA_CSV         = Path("lisa_review.csv")    # from lisa review notebook
TEST_HOLDOUT = 0.10   # fraction of Mapillary train to hold out as test

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT      = Path("dataset/combined_dataset_1280")

# ── Split ratios (must sum to 1.0) ────────────────────────────────────────────
# Only applies to Mapillary — LISA keeps its original Roboflow splits
MAPILLARY_SPLITS = {"train": 0.80, "val": 0.10, "test": 0.10}
MIN_ANNOTATION_PX = 20   # size of smallest annotations to keep (at 1280px) — smaller ones are too small to detect reliably

LISA_SPLITS      = ["train", "val", "test"]   # valid -> val in output

SEED             = 42
random.seed(SEED)

## Step 1 — Build unified class list from both CSVs

### Merge similar classes

In [5]:
# Define merges as {canonical_name: [list of names to fold into it]}
# The canonical name is what all variants will become
MAPILLARY_MERGES = {
    "regulatory--no-turn-on-red--g1": [
        "regulatory--no-turn-on-red--g2",
        "regulatory--no-turn-on-red--g3",
    ],
    "warning--divided-highway-ends--g1": [
        "warning--divided-highway-ends--g2",
    ],
    "complementary--one-direction-left--g1": [
        "complementary--one-direction-right--g1",
    ],
}

def apply_merges(kept_labels, merges):
    """
    Returns a remapped set of kept labels and a
    lookup {old_label: canonical_label} for the converter to use.
    """
    remap = {}
    for canonical, variants in merges.items():
        for variant in variants:
            if variant in kept_labels:
                remap[variant] = canonical
                kept_labels.discard(variant)
            # Also remap canonical to itself so the converter
            # handles it uniformly
        if canonical in kept_labels:
            remap[canonical] = canonical

    # Pass-through for everything not involved in a merge
    for label in kept_labels:
        if label not in remap:
            remap[label] = label

    return kept_labels, remap

In [6]:
def load_mapillary_kept(csv_path):
    """
    Returns set of mapillary label strings that were marked keep=yes.
    """
    kept = set()
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            if row["keep"] == "yes":
                kept.add(row["label"])
    return kept


def load_lisa_mapping(csv_path):
    """
    Returns {lisa_label: mapillary_label} for kept classes.
    """
    mapping = {}
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            if row["keep"] == "yes":
                mapping[row["lisa_label"]] = row["mapillary_label"]
    return mapping

mapillary_kept  = load_mapillary_kept(MAPILLARY_CSV)
lisa_mapping    = load_lisa_mapping(LISA_CSV)

mapillary_kept, mapillary_remap = apply_merges(mapillary_kept, MAPILLARY_MERGES)

# Union of all target label names (Mapillary labels are the canonical names)
all_labels      = sorted(mapillary_kept | set(lisa_mapping.values()))
label_to_id     = {label: i for i, label in enumerate(all_labels)}

print(f"Mapillary keep classes : {len(mapillary_kept)}")
print(f"LISA keep classes      : {len(lisa_mapping)}")
print(f"Unified class count    : {len(all_labels)}")
print()

# Show any LISA classes that introduce a brand new label not in Mapillary
new_from_lisa = set(lisa_mapping.values()) - mapillary_kept
if new_from_lisa:
    print(f"New classes introduced by LISA ({len(new_from_lisa)}):")
    for n in sorted(new_from_lisa):
        print(f"  {n}")
else:
    print("No new classes introduced by LISA — all map to existing Mapillary labels.")

Mapillary keep classes : 55
LISA keep classes      : 21
Unified class count    : 55

No new classes introduced by LISA — all map to existing Mapillary labels.


## Step 2 — Set up output directory structure

In [7]:
for split in ["train", "val", "test"]:
    (OUTPUT_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_ROOT.resolve()}")

Output directory: C:\Projects\TrafficSignRecognition\model_training\dataset\combined_dataset_1280


## Step 3 — Process Mapillary

In [8]:
def load_mapillary_keys(mapillary_root, splits, test_holdout=0.10, seed=42):
    """
    Load and assign each image key to a split.
    Redirects Mapillary's empty test split to val, then carves
    test_holdout fraction of train keys into a proper test set.
    """
    key_to_split = {}
    for split in splits:
        path = mapillary_root / "splits" / f"{split}.txt"
        if not path.exists():
            continue
        with open(path) as f:
            for line in f:
                key = line.strip()
                if key:
                    key_to_split[key] = "val" if split == "test" else split

    train_keys = [k for k, s in key_to_split.items() if s == "train"]
    random.seed(seed)
    random.shuffle(train_keys)
    n_test = int(len(train_keys) * test_holdout)
    for key in train_keys[:n_test]:
        key_to_split[key] = "test"

    from collections import Counter
    counts = Counter(key_to_split.values())
    print(f"  Mapillary split assignment:")
    print(f"    train : {counts['train']:,}")
    print(f"    val   : {counts['val']:,}  (original val + redirected test)")
    print(f"    test  : {counts['test']:,}  ({test_holdout:.0%} held out from train)")

    return key_to_split


def mapillary_to_yolo(bbox, img_w, img_h):
    """Convert Mapillary {xmin,ymin,xmax,ymax} to YOLO cx cy w h (normalised)."""
    xmin, ymin = bbox["xmin"], bbox["ymin"]
    xmax, ymax = bbox["xmax"], bbox["ymax"]
    cx = ((xmin + xmax) / 2) / img_w
    cy = ((ymin + ymax) / 2) / img_h
    w  = (xmax - xmin) / img_w
    h  = (ymax - ymin) / img_h
    return cx, cy, w, h


def is_too_small(bbox, min_px=20):
    """Returns True if the annotation is smaller than min_px in either dimension."""
    w_px = bbox["xmax"] - bbox["xmin"]
    h_px = bbox["ymax"] - bbox["ymin"]
    return w_px < min_px or h_px < min_px


def process_mapillary(mapillary_root, kept_labels, label_to_id,
                      split_ratios, output_root, mapillary_remap=None, MIN_ANNOTATION_PX=32):
    ann_dir = mapillary_root / "annotations"
    img_dir = mapillary_root / "images"

    key_to_split = load_mapillary_keys(
        mapillary_root, ["train", "val", "test"],
        test_holdout=TEST_HOLDOUT,
        seed=SEED,
    )
    split_remap = {"train": "train", "val": "val", "test": "test"}

    if mapillary_remap is None:
        mapillary_remap = {label: label for label in kept_labels}

    stats             = defaultdict(lambda: defaultdict(int))
    skipped_no_image  = 0
    skipped_no_labels = 0
    skipped_pano      = 0
    skipped_too_small = 0
    skipped_occluded  = 0
    written           = 0

    ann_files = list(ann_dir.glob("*.json"))

    for ann_path in tqdm(ann_files, desc="Mapillary"):
        key = ann_path.stem

        img_path = img_dir / f"{key}.jpg"
        if not img_path.exists():
            skipped_no_image += 1
            continue

        with open(ann_path) as f:
            ann = json.load(f)

        # Skip panoramic images — distorted appearance won't match dashcam footage
        if ann.get("ispano", False):
            skipped_pano += 1
            continue

        img_w = ann["width"]
        img_h = ann["height"]

        yolo_lines = []
        for obj in ann.get("objects", []):
            raw_label = obj.get("label", "")
            if raw_label not in mapillary_remap:
                continue

            props = obj.get("properties", {})
            if props.get("ambiguous") or props.get("out-of-frame") or props.get("dummy"):
                continue

            # Skip occluded annotations
            if props.get("occluded"):
                skipped_occluded += 1
                continue

            # Skip annotations too small to detect reliably at 640px
            if is_too_small(obj["bbox"], MIN_ANNOTATION_PX):
                skipped_too_small += 1
                continue

            label    = mapillary_remap[raw_label]
            class_id = label_to_id[label]
            cx, cy, w, h = mapillary_to_yolo(obj["bbox"], img_w, img_h)

            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            w  = max(0.0, min(1.0, w))
            h  = max(0.0, min(1.0, h))

            yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            split_out = split_remap.get(key_to_split.get(key, "train"), "train")
            stats[split_out][label] += 1

        if not yolo_lines:
            skipped_no_labels += 1
            continue

        split_out = split_remap.get(key_to_split.get(key, "train"), "train")

        out_img = output_root / split_out / "images" / f"mply_{key}.jpg"
        shutil.copy2(img_path, out_img)

        out_lbl = output_root / split_out / "labels" / f"mply_{key}.txt"
        with open(out_lbl, "w") as f:
            f.write("\n".join(yolo_lines))

        written += 1

    print(f"  Written            : {written:,} images")
    print(f"  Skipped (no image) : {skipped_no_image:,}")
    print(f"  Skipped (pano)     : {skipped_pano:,}")
    print(f"  Skipped (no kept labels) : {skipped_no_labels:,}")
    print(f"  Annotations dropped (too small) : {skipped_too_small:,}")
    print(f"  Annotations dropped (occluded)  : {skipped_occluded:,}")
    return stats


print("Processing Mapillary...")
mapillary_stats = process_mapillary(
    MAPILLARY_ROOT, mapillary_kept, label_to_id,
    MAPILLARY_SPLITS, OUTPUT_ROOT,
    mapillary_remap=mapillary_remap
)
print("Done.")

Processing Mapillary...
  Mapillary split assignment:
    train : 32,931
    val   : 15,864  (original val + redirected test)
    test  : 3,658  (10% held out from train)


Mapillary:   0%|          | 0/41909 [00:00<?, ?it/s]

  Written            : 8,345 images
  Skipped (no image) : 0
  Skipped (pano)     : 941
  Skipped (no kept labels) : 32,623
  Annotations dropped (too small) : 7,857
  Annotations dropped (occluded)  : 1,470
Done.


## Step 4 — Process LISA

In [9]:
def load_lisa_id_to_name(lisa_root):
    with open(lisa_root / "data.yaml") as f:
        data = yaml.safe_load(f)
    names = data["names"]
    if isinstance(names, dict):
        return {int(k): v for k, v in names.items()}
    return {i: n for i, n in enumerate(names)}


def process_lisa(lisa_root, lisa_mapping, label_to_id,
                 lisa_splits, output_root):
    """
    lisa_mapping: {lisa_label: unified_label}
    """
    lisa_id_to_name = load_lisa_id_to_name(lisa_root)
    split_remap = {"train": "train", "valid": "val", "val": "val", "test": "test"}

    stats = defaultdict(lambda: defaultdict(int))
    skipped_no_image  = 0
    skipped_no_labels = 0
    written = 0

    for split in lisa_splits:
        labels_dir = lisa_root / split / "labels"
        images_dir = lisa_root / split / "images"
        split_out  = split_remap.get(split, split)

        if not labels_dir.exists():
            print(f"  [warn] not found: {labels_dir}")
            continue

        label_files = list(labels_dir.glob("*.txt"))

        for label_file in tqdm(label_files, desc=f"LISA {split}"):
            # Find matching image
            img_path = None
            for ext in [".jpg", ".jpeg", ".png"]:
                candidate = images_dir / (label_file.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break

            if img_path is None:
                skipped_no_image += 1
                continue

            yolo_lines = []
            with open(label_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue

                    lisa_class_id = int(parts[0])
                    lisa_name     = lisa_id_to_name.get(lisa_class_id)
                    if lisa_name is None or lisa_name not in lisa_mapping:
                        continue  # not a kept class

                    unified_label = lisa_mapping[lisa_name]
                    class_id      = label_to_id[unified_label]

                    cx, cy, w, h  = [float(x) for x in parts[1:5]]
                    cx = max(0.0, min(1.0, cx))
                    cy = max(0.0, min(1.0, cy))
                    w  = max(0.0, min(1.0, w))
                    h  = max(0.0, min(1.0, h))

                    yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                    stats[split_out][unified_label] += 1

            if not yolo_lines:
                skipped_no_labels += 1
                continue

            # Copy image — prefix with lisa_ to avoid key collisions with Mapillary
            out_img = output_root / split_out / "images" / f"lisa_{img_path.name}"
            shutil.copy2(img_path, out_img)

            out_lbl = output_root / split_out / "labels" / f"lisa_{label_file.stem}.txt"
            with open(out_lbl, "w") as f:
                f.write("\n".join(yolo_lines))

            written += 1

    print(f"  Written  : {written:,} images")
    print(f"  Skipped (no image)       : {skipped_no_image:,}")
    print(f"  Skipped (no kept labels) : {skipped_no_labels:,}")
    return stats


print("Processing LISA...")
lisa_stats = process_lisa(
    LISA_ROOT, lisa_mapping, label_to_id,
    LISA_SPLITS, OUTPUT_ROOT
)
print("Done.")

Processing LISA...


LISA train:   0%|          | 0/7956 [00:00<?, ?it/s]

LISA val:   0%|          | 0/1018 [00:00<?, ?it/s]

LISA test:   0%|          | 0/838 [00:00<?, ?it/s]

  Written  : 6,003 images
  Skipped (no image)       : 0
  Skipped (no kept labels) : 3,809
Done.


## Step 5 — Write data.yaml

In [10]:
data_yaml = {
    "path":  str(OUTPUT_ROOT.resolve()),
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    len(all_labels),
    "names": all_labels,
}

yaml_out = OUTPUT_ROOT / "data.yaml"
with open(yaml_out, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"data.yaml written to {yaml_out}")

data.yaml written to dataset\combined_dataset_1280\data.yaml


## Step 6 — Summary

In [11]:
import numpy as np

def count_files(path):
    return len(list(path.glob("*"))) if path.exists() else 0

print("=" * 70)
print("COMBINED DATASET SUMMARY")
print("=" * 70)
print(f"  Total classes : {len(all_labels)}")
print()

for split in ["train", "val", "test"]:
    n = count_files(OUTPUT_ROOT / split / "images")
    print(f"  {split:<6} images : {n:,}")

print()

# Per-class annotation counts across all splits and both sources
combined_counts = defaultdict(int)
for stats in [mapillary_stats, lisa_stats]:
    for split_counts in stats.values():
        for label, count in split_counts.items():
            combined_counts[label] += count

arr = np.array([combined_counts.get(l, 0) for l in all_labels])

print(f"  Total annotations : {arr.sum():,}")
print(f"  Mean per class    : {arr.mean():,.1f}")
print(f"  Median per class  : {np.median(arr):,.1f}")
print(f"  Classes < 100     : {(arr < 100).sum()}")
print(f"  Classes < 500     : {(arr < 500).sum()}")
print()

print(f"  {'Class':<55} {'Total':>7}  {'Mapillary':>10}  {'LISA':>6}")
print("  " + "─" * 82)

# Flatten per-source totals
mply_totals = defaultdict(int)
for split_counts in mapillary_stats.values():
    for label, count in split_counts.items():
        mply_totals[label] += count

lisa_totals = defaultdict(int)
for split_counts in lisa_stats.values():
    for label, count in split_counts.items():
        lisa_totals[label] += count

for label in all_labels:
    total = combined_counts.get(label, 0)
    mply  = mply_totals.get(label, 0)
    lisa  = lisa_totals.get(label, 0)
    flag  = "  ⚠️ <100" if total < 100 else ""
    print(f"  {label:<55} {total:>7,}  {mply:>10,}  {lisa:>6,}{flag}")

print("=" * 70)

COMBINED DATASET SUMMARY
  Total classes : 55

  train  images : 11,384
  val    images : 1,652
  test   images : 1,312

  Total annotations : 18,122
  Mean per class    : 329.5
  Median per class  : 192.0
  Classes < 100     : 18
  Classes < 500     : 45

  Class                                                     Total   Mapillary    LISA
  ──────────────────────────────────────────────────────────────────────────────────
  complementary--both-directions--g1                           56          56       0  ⚠️ <100
  complementary--chevron-left--g1                             621         621       0
  complementary--chevron-right--g1                            571         571       0
  complementary--keep-left--g1                                126         126       0
  complementary--keep-right--g1                               468          24     444
  complementary--obstacle-delineator--g1                       97          97       0  ⚠️ <100
  complementary--obstacle-delineator--

# Training

In [1]:
from ultralytics import YOLO
from pathlib import Path
import os

PROJECT_DIR = os.path.abspath("experiments/")
RUN_NAME    = "baseline_v1_1280"

model = YOLO("yolov8s.pt")  # small variant — better than nano for 57 classes

results = model.train(
    # ── Core ──────────────────────────────────────────────────────
    data    = "dataset/combined_dataset_1280/data.yaml",
    epochs  = 100,
    patience = 20,           # early stopping — stops if no improvement for 20 epochs
    imgsz   = 1280,
    batch   = 8,
    device  = 0,             # GPU; use "cpu" if no GPU available
    resume = True,

    # ── Output ────────────────────────────────────────────────────
    project = PROJECT_DIR,
    name    = RUN_NAME,
    exist_ok = False,        # set True to overwrite a previous run with same name

    # ── Optimizer ─────────────────────────────────────────────────
    optimizer = "SGD",       # SGD generalizes better than Adam for detection
    lr0       = 0.01,        # initial learning rate
    lrf       = 0.01,        # final lr = lr0 * lrf (cosine decay target)
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs   = 3.0,
    warmup_momentum = 0.8,
    cos_lr    = True,        # cosine LR schedule — smoother decay than step

    # ── Augmentation ──────────────────────────────────────────────
    hsv_h     = 0.015,       # hue jitter — helps with varied lighting
    hsv_s     = 0.7,         # saturation jitter
    hsv_v     = 0.4,         # brightness jitter — important for day/night variation
    degrees   = 5.0,         # small rotation — signs are mostly upright
    translate = 0.1,
    scale     = 0.5,         # scale jitter — critical for far/close sign detection
    shear     = 2.0,
    perspective = 0.0001,    # slight perspective for dashcam angle variation
    flipud    = 0.0,         # don't flip vertically — signs aren't upside down
    fliplr    = 0.0,         # don't flip horizontally — no/left/right turns would become wrong
    mosaic    = 1.0,         # combine 4 images — helps with small/far signs
    mixup     = 0.1,         # blend two images — improves generalization
    copy_paste = 0.1,        # paste objects into scenes — helps rare classes

    # ── Evaluation ────────────────────────────────────────────────
    val       = True,
    save      = True,
    save_period = 10,        # save checkpoint every 10 epochs
    plots     = True,        # generate confusion matrix and metric plots
)

Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, crop_fraction=1.0, cutmix=0.0, data=dataset/combined_dataset_1280/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=False, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, label_smoothing=0.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8s, nbs=64, nms=False, opset=17, opt

AssertionError: yolov8s.pt training to 500 epochs is finished, nothing to resume.
Start a new training without resuming, i.e. 'yolo train model=yolov8s.pt'

In [1]:
from ultralytics import YOLO

model = YOLO("experiments/baseline_v1_1280/weights/best.pt")

metrics = model.val(
    data="dataset/combined_dataset_1280/data.yaml",
    split="test",
    imgsz=1280,
    device=0,
    plots=True,
    save_json=True,
)

print(f"mAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
Model summary (fused): 73 layers, 11,146,869 parameters, 0 gradients, 28.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 163.6151.2 MB/s, size: 689.8 KB)
val: Scanning C:\Projects\TrafficSignRecognition\model_training\dataset\combined_dataset_1280\test\labels... 1312 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1312/1312 948.2it/s 1.4s0.1s
val: New cache created: C:\Projects\TrafficSignRecognition\model_training\dataset\combined_dataset_1280\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 82/82 3.1it/s 26.2s0.3ss
                   all       1312       1613       0.83      0.825       0.88      0.706
complementary--both-directions--g1          8          8      0.883      0.946      0.967       0.67
complementary--chevron-left--g1         34         66      0.949      0.846      0.956      